# Customer Support Chatbot - Training Notebook

## Project Overview
Build an intelligent customer support chatbot with:
- Intent classification (>90% accuracy)
- Natural response generation
- 15+ supported intents
- Streamlit deployment

**Total Time:** 3-4 hours

## 1. Load Intent Data

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Load intents
with open('intents.json', 'r') as f:
    intents_data = json.load(f)

print(f"✓ Loaded {len(intents_data['intents'])} intents")

# Display intent overview
print("\nAvailable Intents:")
for intent in intents_data['intents']:
    print(f"  • {intent['tag']}: {len(intent['patterns'])} patterns, {len(intent['responses'])} responses")

## 2. Prepare Training Data

In [ ]:
# Extract patterns and labels
patterns = []
labels = []

for intent in intents_data['intents']:
    for pattern in intent['patterns']:
        patterns.append(pattern)
        labels.append(intent['tag'])

print(f"Total training samples: {len(patterns)}")
print(f"Number of classes: {len(set(labels))}")

# Create DataFrame
df = pd.DataFrame({'text': patterns, 'intent': labels})

# Display class distribution
print("\nClass Distribution:")
print(df['intent'].value_counts())

# Visualize
plt.figure(figsize=(14, 6))
df['intent'].value_counts().plot(kind='barh', color='skyblue')
plt.xlabel('Number of Patterns', fontsize=11)
plt.ylabel('Intent', fontsize=11)
plt.title('Intent Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and preprocess text."""
    # Lowercase
    text = text.lower()
    
    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
df['text_clean'] = df['text'].apply(preprocess_text)

print("Preprocessing Examples:")
for i in range(3):
    print(f"\nOriginal: {df['text'].iloc[i]}")
    print(f"Cleaned:  {df['text_clean'].iloc[i]}")

## 4. Feature Extraction with TF-IDF

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], df['intent'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['intent']
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words='english'
)

# Fit and transform
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"\nTF-IDF feature shape: {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

## 5. Train Intent Classifier

In [ ]:
# Train Logistic Regression classifier
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='ovr'
)

classifier.fit(X_train_tfidf, y_train)

# Make predictions
y_pred = classifier.predict(X_test_tfidf)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Intent Classification Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 6. Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
intent_labels = sorted(df['intent'].unique())

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=intent_labels,
            yticklabels=intent_labels,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Intent', fontsize=11)
plt.ylabel('True Intent', fontsize=11)
plt.title('Intent Classification: Confusion Matrix', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Build Response Generator

In [ ]:
import random

class ChatbotResponder:
    """Generates responses based on predicted intent."""
    
    def __init__(self, intents_data):
        self.intents = {intent['tag']: intent['responses'] 
                       for intent in intents_data['intents']}
    
    def get_response(self, intent):
        """Get random response for predicted intent."""
        if intent in self.intents:
            return random.choice(self.intents[intent])
        else:
            return "I'm not sure how to help with that. Could you rephrase?"

# Create responder
responder = ChatbotResponder(intents_data)

print("✓ Response generator created")

## 8. Complete Chatbot Pipeline

In [ ]:
def chatbot_predict(user_input):
    """
    Complete chatbot pipeline: input → intent → response.
    
    Args:
        user_input (str): User's message
    
    Returns:
        dict: Intent, confidence, and response
    """
    # Preprocess
    cleaned = preprocess_text(user_input)
    
    # Vectorize
    vectorized = vectorizer.transform([cleaned])
    
    # Predict intent
    intent = classifier.predict(vectorized)[0]
    
    # Get confidence
    probabilities = classifier.predict_proba(vectorized)[0]
    confidence = max(probabilities)
    
    # Generate response
    response = responder.get_response(intent)
    
    return {
        'intent': intent,
        'confidence': confidence,
        'response': response
    }

print("✓ Chatbot pipeline ready!")

## 9. Test the Chatbot

In [ ]:
# Test queries
test_queries = [
    "Hi there!",
    "Where is my package?",
    "I want to return this product",
    "My payment didn't work",
    "What are your hours?",
    "Do you have any promo codes?",
    "Thank you so much!",
    "I need to cancel my order"
]

print("CHATBOT TEST CONVERSATIONS:\n")
print("=" * 80)

for query in test_queries:
    result = chatbot_predict(query)
    
    print(f"\nUser: {query}")
    print(f"Intent: {result['intent']} (Confidence: {result['confidence']:.1%})")
    print(f"Bot: {result['response']}")
    print("-" * 80)

## 10. Save Models

In [ ]:
import joblib

# Save classifier and vectorizer
joblib.dump(classifier, 'intent_classifier.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("✓ Models saved:")
print("  • intent_classifier.pkl")
print("  • tfidf_vectorizer.pkl")
print("\nUse these files in the Streamlit app!")

## Success Criteria

- ✅ Intent classifier accuracy >90%
- ✅ 15+ intents supported
- ✅ Natural response generation
- ✅ Complete chatbot pipeline working
- ✅ Models saved for deployment

**Next:** Deploy with Streamlit!